# 2 つのコーパスを比べて特徴語を見つける（対数尤度比）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_corpus_statistics/keyness.ipynb)

NLP クックブック 05-11 ／ コーパスの統計と可視化

コーパス A にはコーパス B と比べてどんな語が有意に多いか、を統計的に取り出したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, math, re, urllib.request, zipfile
from collections import Counter
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()

def count(url):
    return Counter((w.feature.lemma or w.surface).split("-")[0] for line in aozora(url).split("\n") for w in tagger(line) if w.feature.pos1 != "補助記号")

A = count("https://www.aozora.gr.jp/cards/000148/files/752_ruby_2438.zip")  # 坊っちゃん (夏目漱石)
B = count("https://www.aozora.gr.jp/cards/000081/files/456_ruby_145.zip")   # 銀河鉄道の夜 (宮沢賢治)

In [3]:
def log_likelihood(a, b, na, nb):
    """Dunning の対数尤度比 G2。3.84 以上で 5% 水準、15.13 以上で 0.01% 水準で有意"""
    e1, e2 = na * (a + b) / (na + nb), nb * (a + b) / (na + nb)
    return 2 * ((a * math.log(a / e1) if a else 0) + (b * math.log(b / e2) if b else 0))

na, nb = sum(A.values()), sum(B.values())
rows = []
for w in set(A) | set(B):
    g2 = log_likelihood(A[w], B[w], na, nb)
    rows.append((g2 if A[w] / na > B[w] / nb else -g2, w, A[w], B[w]))
rows.sort()

print("坊っちゃんに特徴的:", [(w, a, b) for _, w, a, b in rows[::-1][:15]])
print("銀河鉄道の夜に特徴的:", [(w, a, b) for _, w, a, b in rows[:15]])

坊っちゃんに特徴的: [('俺', 477, 1), ('と', 1748, 380), ('ヤマアラシ', 155, 0), ('ない', 633, 93), ('赤', 177, 2), ('君', 185, 5), ('シャツ', 170, 4), ('事', 312, 39), ('無い', 424, 72), ('ば', 125, 5), ('物', 296, 41), ('野', 89, 1), ('有る', 399, 68), ('遣る', 160, 13), ('校長', 70, 0)]
銀河鉄道の夜に特徴的: [('ます', 194, 551), ('ジョバンニ', 0, 189), ('カムパネルラ', 0, 101), ('です', 211, 296), ('様', 193, 223), ('の', 2428, 1389), ('其の', 109, 156), ('達', 4, 52), ('そして', 0, 41), ('よ', 28, 82), ('もう', 55, 103), ('小さな', 2, 39), ('僕', 47, 87), ('ああ', 5, 40), ('川', 2, 34)]


- 内容語だけでなく機能語も上位に来る。坊っちゃんでは一人称「おれ」が見出し語の「俺」（477 対 1）として 1 位に、銀河鉄道の夜では文末の「ます」「です」が上位に出る。文体の違いの分析に使える。
- UniDic の見出し語で数えるので、人名の「山嵐」は「ヤマアラシ」（155 対 0）として出る。「赤シャツ」は「赤」と「シャツ」に、「野だ」は「野」に分かれる。
- カイ二乗検定より低頻度語に頑健なので、コーパス言語学ではこの指標 (keyness) が標準的に使われる。